In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, mne
from mne.time_frequency import psd_array_welch
from src.load import list_children, load_child
from src.channels import CHANNELS, SFREQ

N = 4 * SFREQ                         # samples per 4 s piece = 512
pieces, child, label = [], [], []
for _, k in pd.DataFrame(list_children()).iterrows():
    x = mne.filter.filter_data(load_child(k.path), SFREQ, 1.0, 40.0, verbose=False)
    x = x[:, :60 * SFREQ]                               # keep the first 60 s only
    pieces.append(x.reshape(19, 15, N).transpose(1, 0, 2))   # -> (15 pieces, 19 ch, 512)
    child += [k.child_id] * 15
    label += [k.label] * 15

W = np.concatenate(pieces)            # all pieces: (1815, 19, 512)
child, label = np.array(child), np.array(label)
print(W.shape)

ImportError: cannot import name 'list_children' from 'src.load' (c:\Users\Rajesh Sampathkumar\Documents\GitHub\eeg-adhd-honest-eval\notebooks\..\src\load.py)

In [4]:
import torch
from torch import nn
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from src.eegnet import EEGNet

X = torch.tensor(W / 100, dtype=torch.float32).unsqueeze(1)   # scale µV to ~±1, add a "1" axis
Y = torch.tensor(label)

def run_fold(train, test, epochs=30, seed=0):
    torch.manual_seed(seed)
    net = EEGNet()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        net.train()
        order = np.random.default_rng(seed + ep).permutation(train)
        for i in range(0, len(order), 64):                 # mini-batches of 64 pieces
            b = order[i:i + 64]
            opt.zero_grad()
            loss_fn(net(X[b]), Y[b]).backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        return torch.softmax(net(X[test]), dim=1)[:, 1].numpy()   # P(ADHD) per test piece

def evaluate(cv):
    p = np.zeros(len(Y))
    for f, (train, test) in enumerate(cv.split(W, label, groups=child)):
        p[test] = run_fold(train, test)
        print("  fold", f + 1, "done")
    votes = pd.DataFrame({"child": child, "y": label, "p": p}).groupby("child").mean()
    return roc_auc_score(label, p), roc_auc_score(votes.y, votes.p)

for name, cv in [("CHEAT", StratifiedKFold(5, shuffle=True, random_state=42)),
                 ("HONEST", StratifiedGroupKFold(5, shuffle=True, random_state=42))]:
    piece_auc, child_auc = evaluate(cv)
    print(f"{name:7s} piece AUC {piece_auc:.2f}   per-child AUC {child_auc:.2f}")

c:\Users\Rajesh Sampathkumar\miniconda3\envs\medimg\Lib\site-packages\sklearn\model_selection\_split.py:884: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
  fold 5 done
CHEAT   piece AUC 0.92   per-child AUC 0.98
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done
  fold 5 done
HONEST  piece AUC 0.73   per-child AUC 0.77


In [1]:
inner = [i for i, c in enumerate(CHANNELS) if c not in ["Fp1", "Fp2", "F7", "F8", "T7", "T8"]]

def subject_wise_auc(Xdata, seed):
    p = np.zeros(len(Y))
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    for train, test in cv.split(W, label, groups=subject):
        torch.manual_seed(seed)
        net = EEGNet(n_ch=Xdata.shape[2])
        opt, loss_fn = torch.optim.Adam(net.parameters(), lr=1e-3), nn.CrossEntropyLoss()
        for ep in range(30):
            net.train()
            order = np.random.default_rng(seed + ep).permutation(train)
            for i in range(0, len(order), 64):
                b = order[i:i + 64]
                opt.zero_grad(); loss_fn(net(Xdata[b]), Y[b]).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            p[test] = torch.softmax(net(Xdata[test]), dim=1)[:, 1].numpy()
    agg = pd.DataFrame({"subject": subject, "y": label, "p": p}).groupby("subject").mean()
    return roc_auc_score(agg.y, agg.p)

for name, Xdata in [("all 19 channels", X), ("13 inner channels (no EOG/EMG sites)", X[:, :, inner, :])]:
    aucs = [subject_wise_auc(Xdata, s) for s in [1, 2, 3]]
    print(f"{name:38s} subject-level AUC {np.mean(aucs):.2f} ± {np.std(aucs):.2f}  {np.round(aucs, 2)}")

NameError: name 'CHANNELS' is not defined